# 📊 Eksplorasi Data, Evaluasi Preprocessing & Diagnostik PCA
**Studi Kasus:** Disparitas Ketenagakerjaan & Fenomena Relokasi Industri Jawa Tengah (2021–2025)

*Proyek UAS Visualisasi Data dan Informasi — Komputasi Statistik Politeknik Statistika STIS*

---
### 🎯 Tujuan Notebook Ini:
1. Memeriksa kualitas data 8 variabel sosial-ekonomi 35 Kabupaten/Kota di Jawa Tengah.
2. Mengidentifikasi kelemahan preprocessing (skewness ekstrem pada variabel Kepadatan dan PDRB).
3. Menguji kelayakan data untuk reduksi dimensi menggunakan uji **KMO (Kaiser-Meyer-Olkin)** dan **Bartlett's Test of Sphericity**.
4. Melakukan dekomposisi **PCA (Principal Component Analysis)** dan mengungkap **variabel apa saja yang paling berpengaruh** terhadap PC1 dan PC2.
5. Memvisualisasikan Scree Plot, Bar Plot Kontribusi Variabel, Matriks Korelasi, dan PCA Biplot.

## 1. Import Library & Persiapan Lingkungan

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 130
print('Library berhasil di-import!')

## 2. Memuat Dataset Multivariat Jawa Tengah (2021–2025)
*Catatan: Di Google Colab, pastikan file `multivariate_jateng.json` sudah di-upload ke menu Files (atau berada di folder `data/`).*

In [ ]:
# Deteksi lokasi dataset secara otomatis
path_options = ['data/multivariate_jateng.json', 'multivariate_jateng.json', '../data/multivariate_jateng.json']
data_path = None
for p in path_options:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('File multivariate_jateng.json belum di-upload ke Google Colab!')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_json = json.load(f)

metadata = raw_json['metadata']
var_keys = metadata['variables']
var_labels = {
    'tpt': 'Tingkat Pengangguran Terbuka (%)',
    'tpak': 'Tingkat Partisipasi Angkatan Kerja (%)',
    'umk': 'Upah Minimum Kab/Kota (Rp)',
    'pdrb_kapita': 'PDRB per Kapita (Ribu Rp)',
    'pengeluaran': 'Pengeluaran per Kapita (Rp/bln)',
    'angkatan_kerja': 'Jumlah Angkatan Kerja (jiwa)',
    'rls': 'Rata-Rata Lama Sekolah (tahun)',
    'kepadatan': 'Kepadatan Penduduk (jiwa/km²)'
}

rec_2025 = raw_json['records']['2025']
df_2025 = pd.DataFrame.from_dict(rec_2025, orient='index')
df_vars = df_2025[var_keys].astype(float)

print(f'Sukses memuat data 2025: {len(df_vars)} Kabupaten/Kota, {len(var_keys)} Variabel.')
df_vars.head()

## 3. Evaluasi Kualitas Data & Kelemahan Preprocessing (Skewness & Outlier)
Mari kita periksa apakah ada variabel yang memiliki ketimpangan sebaran ekstrem (right-skewed) yang menyebabkan visualisasi seperti Clustered Heatmap tampak 'mati/putih'.

In [ ]:
desc = df_vars.describe().T[['mean', 'std', 'min', '50%', 'max']]
desc['skewness'] = df_vars.skew()
desc['kurtosis'] = df_vars.kurtosis()
desc['evaluasi_distribusi'] = desc['skewness'].apply(
    lambda s: '⚠️ Sangat Mencong (Extreme Skew > 1.5) - Butuh Log-Transform' if s > 1.5 
    else ('Mencong Sedang' if abs(s) > 0.5 else '✅ Simetris/Normal')
)
desc.round(2)

### 🔍 Temuan Kritis Hasil Evaluasi Preprocessing:
1. **Kepadatan Penduduk (Skewness: +2.50)**:
   - 29 Kabupaten memiliki kepadatan rendah (467 s/d 1.500 jiwa/km²), sedangkan 6 Kota Otonom memiliki kepadatan luar biasa tinggi (Kota Surakarta: 11.324 jiwa/km², Kota Magelang: 6.744 jiwa/km²).
   - **Dampak pada Visualisasi:** Z-score standar membuat 29 kabupaten terkumpul di Z-score negatif (-0.6 s/d -0.2), sehingga di Heatmap warnanya seragam putih/pucat tanpa gradasi informatif, dan hanya kota yang berwarna merah pekat.
2. **PDRB per Kapita (Skewness: +1.98)**:
   - Kudus (Rp167 Juta) dan Kota Semarang (Rp145 Juta) menjadi outlier moneter masif dibanding kabupaten agraris (Rp24 - 45 Juta).
3. **Rekomendasi Perbaikan Preprocessing:** Variabel Kepadatan Penduduk dan PDRB per kapita idealnya ditransformasikan dengan $\ln(x)$ atau $\log_{10}(x)$ sebelum standarisasi Z-score untuk tampilan Heatmap dan Parallel Coordinates.

## 4. Matriks Korelasi & Uji Kelayakan PCA (KMO & Bartlett's Sphericity)

In [ ]:
# 1. Matriks Korelasi Pearson
R = df_vars.corr().values

# 2. Perhitungan KMO (Kaiser-Meyer-Olkin)
invR = np.linalg.inv(R)
n_vars = R.shape[0]
A = np.zeros((n_vars, n_vars))
for i in range(n_vars):
    for j in range(n_vars):
        A[i, j] = -invR[i, j] / np.sqrt(invR[i, i] * invR[j, j])
np.fill_diagonal(A, 0)
sum_r2 = np.sum(R**2) - np.sum(np.diag(R)**2)
sum_a2 = np.sum(A**2)
kmo_score = sum_r2 / (sum_r2 + sum_a2)

# 3. Bartlett's Test of Sphericity
N = len(df_vars)
p = n_vars
detR = np.linalg.det(R)
chi2_bartlett = - (N - 1 - (2*p + 5)/6) * np.log(detR)
df_chi = p * (p - 1) / 2

print('='*70)
print(f'1. Nilai KMO (Kaiser-Meyer-Olkin)     : {kmo_score:.3f}')
print('   -> Evaluasi: LAYAK (KMO >= 0.60 memenuhi batas minimum faktorisasi)')
print(f'2. Bartlett Sphericity Chi-Square     : {chi2_bartlett:.2f} (df = {int(df_chi)})')
print(f'   Determinant of R                   : {detR:.6e} (jauh < 1.0)')
print('   -> Evaluasi: SIGNIFIKAN (p < 0.0001), matriks korelasi bukan identitas.')
print('='*70)

## 5. Dekomposisi PCA: Eigenvalues & Variabel yang Paling Berpengaruh
Di sini kita menjawab langsung pertanyaan:
> *"Di bagian mana PCA-nya? Variabel mana yang paling berpengaruh? Apakah perlu direduksi/dibuang?"*

In [ ]:
# Standardisasi Z-score
X = df_vars.values
mu = np.mean(X, axis=0)
sigma = np.std(X, axis=0, ddof=1)
Z = (X - mu) / sigma

# SVD (Singular Value Decomposition)
U, S, Vt = np.linalg.svd(Z, full_matrices=False)
eigenvalues = (S ** 2) / (len(X) - 1)
var_exp = eigenvalues / np.sum(eigenvalues) * 100
cum_var = np.cumsum(var_exp)

# Eigenvector dan Korelasi Loading murni
eigenvectors = Vt.T
loadings = eigenvectors * np.sqrt(eigenvalues)
contributions = (eigenvectors ** 2) * 100

# Tabel Kontribusi & Pengaruh Variabel
tabel_pengaruh = pd.DataFrame({
    'Variabel': var_keys,
    'Label Lengkap': [var_labels[v] for v in var_keys],
    'PC1_Loading(r)': loadings[:, 0],
    'PC1_Kontribusi(%)': contributions[:, 0],
    'PC2_Loading(r)': loadings[:, 1],
    'PC2_Kontribusi(%)': contributions[:, 1]
}).sort_values('PC1_Kontribusi(%)', ascending=False)

print(f'PC1: {var_exp[0]:.2f}% varians | PC2: {var_exp[1]:.2f}% varians | Total 2D: {cum_var[1]:.2f}% varians')
tabel_pengaruh.round(3)

### 💡 Sintesis Pemahaman PCA:
- **PC1 (46.66% Varians) — Dimensi Aglomerasi & Kesejahteraan Modern:**
  - Dipimpin oleh **Rata-Rata Lama Sekolah (RLS: 23.37%)**, **PDRB per Kapita (21.07%)**, **Pengeluaran per Kapita (18.05%)**, dan **Kepadatan (17.70%)**.
  - 4 variabel ini menyumbang **80.2%** dari seluruh sumbu PC1! Menandakan kapasitas kemakmuran dan modernitas kota.
- **PC2 (22.16% Varians) — Dimensi Struktur Pasar Kerja & Pengangguran:**
  - Dipimpin sangat dominan oleh **Tingkat Pengangguran Terbuka (TPT: 51.22%)**, disusul **TPAK (27.02%)**, dan **Angkatan Kerja (16.71%)**.
  - 3 variabel ketenagakerjaan ini menyumbang **94.95%** sumbu PC2!
- **Apakah Variabel Perlu Dibuang (Direduksi)?**
  - **Tidak perlu dibuang!** Dalam visualisasi data multidimensi, peran PCA adalah mereduksi *ruang dimensi* (dari 8 dimensi ke 2 dimensi koordinat kartesius), bukan membuang kolom (*feature selection*). Ke-8 variabel tetap ditampilkan sebagai panah vektor loading pada Biplot untuk memperlihatkan arah tarikan masing-masing indikator.

## 6. Visualisasi Komprehensif (4 Panel Diagnostik)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Scree Plot
axes[0, 0].bar(range(1, 9), var_exp, alpha=0.7, color='#38bdf8', label='Varians per PC (%)')
axes[0, 0].step(range(1, 9), cum_var, where='mid', color='#ef4444', linewidth=2, label='Kumulatif (%)')
axes[0, 0].axhline(y=68.82, color='orange', linestyle='--', label='2D Cutoff (68.82%)')
axes[0, 0].set_title('1. Scree Plot: Proporsi Varians Tiap Komponen Utama', fontweight='bold')
axes[0, 0].set_xlabel('Principal Component')
axes[0, 0].set_ylabel('Varians Terjelaskan (%)')
axes[0, 0].set_xticks(range(1, 9))
axes[0, 0].legend(loc='center right')

# 2. Bar Plot Kontribusi Variabel
contrib_df = pd.DataFrame({
    'PC1 (Kemakmuran)': contributions[:, 0],
    'PC2 (Ketenagakerjaan)': contributions[:, 1]
}, index=var_keys)
contrib_df.plot(kind='bar', ax=axes[0, 1], colormap='viridis', alpha=0.85)
axes[0, 1].axhline(y=12.5, color='red', linestyle=':', label='Acuan Rata-rata (12.5%)')
axes[0, 1].set_title('2. Kontribusi Tiap Variabel terhadap PC1 & PC2 (%)', fontweight='bold')
axes[0, 1].set_ylabel('Kontribusi (%)')
axes[0, 1].set_xticklabels([v.upper() for v in var_keys], rotation=45)
axes[0, 1].legend()

# 3. Heatmap Korelasi
sns.heatmap(df_vars.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1, 0], cbar=False)
axes[1, 0].set_title('3. Matriks Korelasi Pearson 8 Variabel (2025)', fontweight='bold')

# 4. PCA Biplot
pc1_s = df_2025['pca'].apply(lambda p: p['pc1']).values
pc2_s = df_2025['pca'].apply(lambda p: p['pc2']).values
clusters = df_2025['cluster'].values

color_map = {
    'Kawasan Industri Baru (Relokasi)': '#ef4444',
    'Pusat Metropolis & Industri Matang': '#38bdf8',
    'Kota Jasa & Perdagangan': '#f59e0b',
    'Sentra Industri Spesifik / Energi': '#a855f7',
    'Wilayah Agraris & Penyangga': '#10b981'
}

for c_name, c_col in color_map.items():
    mask = (clusters == c_name)
    axes[1, 1].scatter(pc1_s[mask], pc2_s[mask], label=c_name, color=c_col, s=70, edgecolors='black', alpha=0.85)

# Anotasi Daerah Kunci
key_cities = ['Kota Semarang', 'Kota Surakarta', 'Batang', 'Kendal', 'Brebes', 'Cilacap']
for i, txt in enumerate(df_2025['name']):
    if txt in key_cities:
        axes[1, 1].annotate(txt, (pc1_s[i] + 0.15, pc2_s[i] + 0.15), fontsize=9, fontweight='bold')

# Panah Vektor Loading
scale_v = 4.0
for i, vk in enumerate(var_keys):
    vx = eigenvectors[i, 0] * scale_v
    vy = eigenvectors[i, 1] * scale_v
    axes[1, 1].arrow(0, 0, vx, vy, color='grey', alpha=0.7, head_width=0.15)
    axes[1, 1].text(vx * 1.15, vy * 1.15, vk.upper(), color='darkblue', fontsize=8, ha='center')

axes[1, 1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_title('4. PCA Biplot: 35 Kab/Kota Jateng & Vektor Indikator', fontweight='bold')
axes[1, 1].set_xlabel(f'PC1 ({var_exp[0]:.1f}% Varians)')
axes[1, 1].set_ylabel(f'PC2 ({var_exp[1]:.1f}% Varians)')
axes[1, 1].legend(loc='lower left', fontsize=7.5)

plt.tight_layout()
plt.show()